### Recommender System

Sequential processing and recommendation system.

In [1]:
import pandas as pd
import numpy as np
from pynq import Overlay, allocate
from utils import *
import numpy as np


Load overlay

In [2]:
# Load the FPGA hardware design (bitstream + metadata).
# The file "bd_wrapper_embedding.xsa" was generated by Vivado and contains:
#   - The PL bitstream
#   - Hardware handoff information (address map, IP configuration)
#   - AXI interconnect descriptions used by PYNQ
#
# Loading the Overlay configures the programmable logic (PL) and
# exposes the hardware IP blocks as Python-accessible objects.

ol = Overlay("bd_wrapper_embedding.xsa")

Explore available IP cores

In [3]:
# List all IP cores available in the loaded overlay.
# 'ip_dict' is a dictionary autogenerated by PYNQ, where:
#   - keys   = IP core names (as synthesized in Vivado)
#   - values = metadata (address ranges, driver info, registers, etc.)
#
# This is useful for discovering the names of DMA blocks,AXI-Lite peripherals, and custom accelerators.

list(ol.ip_dict.keys())

['axi_dma_0', 'myproject_embedding_0', 'zynq_ultra_ps_e_0']

Create objects


In [4]:
# Get a reference to the AXI DMA instance defined in the hardware design.
# The name 'axi_dma_0' must match the IP name inside the Vivado block design.
dma = ol.axi_dma_0

# DMA channels:
#   - sendchannel:  transfers data from PS to PL  (input to accelerator)
#   - recvchannel:  transfers data from PL to PS  (output from accelerator)
dma_send = dma.sendchannel
dma_recv = dma.recvchannel

# Get a reference to the custom accelerator IP core.
# 'myproject_embedding_0' is the name Vivado generated for the HLS module.
# This gives access to its AXI-Lite control registers (e.g., start, status).
ip = ol.myproject_embedding_0

IP_CTRL_REG = 0x00  # Control register: writing 1 triggers inference

Load testing dataset

In [5]:
# Load test sequences and their corresponding targets.
# Each row in 'recomender_test_sequences.csv' represents one input sample
# for the FPGA-based recommender model.
# The target CSV contains the expected label under the column "target".

x_test = pd.read_csv("recomender_test_sequences.csv").values
y_test = pd.read_csv("recomender_test_targets.csv")["target"].values

# Print dataset shapes for verification.
# x_test.shape = (num_samples, sequence_length)
# y_test.shape = (num_samples,)
print("Test:", x_test.shape, y_test.shape)

# Number of test samples to process through the accelerator.
N = x_test.shape[0]


Test: (2900, 30) (2900,)


Variables

In [6]:
# Length of each input sequence (number of features per sample).
SEQ_LEN = 30      

# Number of output classes produced by the recommender model.
N_CLASSES = 100    

# Initialize evaluation metrics.
# correct_top1  : number of samples where the top predicted class matches the target
# correct_top5  : number of samples where the target is in the top-5 predictions
# correct_top10 : number of samples where the target is in the top-10 predictions
# mrr           : Mean Reciprocal Rank accumulator
correct_top1 = 0
correct_top5 = 0
correct_top10 = 0
mrr = 0

# Array to store FPGA predictions (logits or scores) for all samples.
# Shape: (N samples, 100 classes)
fpga_preds = np.zeros((N, N_CLASSES), dtype=np.float32)


Define DMA buffers

In [7]:
# Allocate DMA-compatible buffers for communication with the accelerator.
# PYNQ's `allocate()` creates physically contiguous memory that the AXI DMA
# can access directly (required for all DMA transfers).

# Input buffer:
# - Contains one sequence of 30 integer features.
# - dtype=int32 matches the expected input format of the accelerator.
in_buffer = allocate(shape=(SEQ_LEN,), dtype=np.int32)

# Output buffer:
# - The accelerator produces 100 class scores (e.g., fixed-point logits).
# - dtype=int32 matches the AXI output word width and the accelerator's format.
out_buffer = allocate(shape=(N_CLASSES,), dtype=np.int32)


Inference process

In [8]:
for i, sample in enumerate(x_test):
    # Load the current test sample into the DMA input buffer.
    # The input consists of 30 integer features for each sample.
    for t in range(30):
        in_buffer[t] = int(sample[t])
    # Start the inference IP core.
    ip.write(IP_CTRL_REG, 1)
    # Launch DMA transfers:
    #  - sendchannel: host to device
    #  - recvchannel: device to host
    dma.sendchannel.transfer(in_buffer)
    dma.recvchannel.transfer(out_buffer)
    # Wait for the DMA transfers to complete.
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    # Convert FPGA outputs from fixed-point format (scaled by 2^6).
    fpga_logits = np.array(out_buffer, dtype=np.float32) / 64.0
    fpga_preds[i, :] = fpga_logits
    # Periodic progress log
    if i % 100 == 0:
        print(f"[{i}/{len(x_test)}], First logits: {fpga_logits[:5]}")

[0/2900], First logits: [0. 0. 0. 0. 0.]
[100/2900], First logits: [0. 0. 0. 0. 0.]
[200/2900], First logits: [0. 0. 0. 0. 0.]
[300/2900], First logits: [0.296875 0.109375 0.03125  0.109375 0.03125 ]
[400/2900], First logits: [0. 0. 0. 0. 0.]
[500/2900], First logits: [0. 0. 0. 0. 0.]
[600/2900], First logits: [0. 0. 0. 0. 0.]
[700/2900], First logits: [0. 0. 0. 0. 0.]
[800/2900], First logits: [0. 0. 0. 0. 0.]
[900/2900], First logits: [0.4375   0.046875 0.046875 0.15625  0.015625]
[1000/2900], First logits: [0. 0. 0. 0. 0.]
[1100/2900], First logits: [0. 0. 0. 0. 0.]
[1200/2900], First logits: [0. 0. 0. 0. 0.]
[1300/2900], First logits: [0.21875  0.078125 0.078125 0.078125 0.015625]
[1400/2900], First logits: [0. 0. 0. 0. 0.]
[1500/2900], First logits: [0. 0. 0. 0. 0.]
[1600/2900], First logits: [0. 0. 0. 0. 0.]
[1700/2900], First logits: [0. 0. 0. 0. 0.]
[1800/2900], First logits: [0.046875 0.046875 0.046875 0.046875 0.046875]
[1900/2900], First logits: [0. 0. 0. 0. 0.]
[2000/2900],

In [9]:
# Compute evaluation metrics for FPGA predictions.
# `fpga_preds` : array of shape (N_samples, N_CLASSES) containing logits/scores
# `y_test`     : true class labels (integer class IDs)

# Top-1 accuracy: the model's most confident prediction must match the target.
top1 = top_k_accuracy(fpga_preds, y_test, k=1)

# Top-5 accuracy: the correct label must appear among the 5 highest scores.
top5 = top_k_accuracy(fpga_preds, y_test, k=5)

# Top-10 accuracy: same idea, but allowing 10 highest-scoring classes.
top10 = top_k_accuracy(fpga_preds, y_test, k=10)

# Mean Reciprocal Rank at 10:
# 1/rank of the correct label, but only counting ranks up to K=10.
mrr10 = mrr_at_k(fpga_preds, y_test, k=10)

# Display all metrics.
print(f"{'Metric':<10} {'Keras':>10} {'FPGA':>10} {'Difference':>12}")
print("-" * 44)
print(f"{'Top-1':<10} {0.1972:>10.4f} {top1:>10.4f} {top1 - 0.1972:>+12.4f}")
print(f"{'Top-5':<10} {0.7072:>10.4f} {top5:>10.4f} {top5 - 0.7072:>+12.4f}")
print(f"{'Top-10':<10} {0.9910:>10.4f} {top10:>10.4f} {top10 - 0.9910:>+12.4f}")
print(f"{'MRR@10':<10} {0.4022:>10.4f} {mrr10:>10.4f} {mrr10 - 0.4022:>+12.4f}")


Metric          Keras       FPGA   Difference
--------------------------------------------
Top-1          0.1972     0.1879      -0.0093
Top-5          0.7072     0.6269      -0.0803
Top-10         0.9910     0.9521      -0.0389
MRR@10         0.4022     0.3792      -0.0230



----

This work was supported in part by the [AMD University Program](https://www.amd.com/en/corporate/university-program.html) 